# Sign / lane / kerb pass on a Colab GPU

Runs `bikesafe.infrastructure` (street-scene segmenter, sign typing and reading) for the rides that
`scripts/colab_bundle.py` packed into `MyDrive/bikesafe`, with the
same settings as the laptop run, and writes the results to `MyDrive/bikesafe/work/analysis/<ride>/` on Drive.

Videos are read from `MyDrive/bikesafe/videos/` or from the laptop folder `bikesafe_videos` that Google Drive for
desktop backs up (it appears under *Other computers*).

1. *Runtime > Change runtime type*: a GPU (L4 or A100 if offered; T4 works, slower).
2. *Runtime > Run all*, and allow access to Google Drive.
3. Keep this tab open. If the runtime disconnects, run all again: each ride resumes from its last checkpoint
   (saved on Drive every 100 samples) and finished rides are skipped.
4. Each ride starts as soon as its video has finished uploading; the last cell says `complete: True` for each
   ride when everything is done.
5. Back on the laptop: `python scripts/colab_bundle.py --fetch "G:/My Drive/bikesafe"`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'NO GPU: Runtime > Change runtime type > GPU')

In [ ]:
import json, os, pathlib, shutil, sys
DRIVE = pathlib.Path('/content/drive/MyDrive/bikesafe')
ROOT = pathlib.Path('/content/bikesafe')  # local copy of the code; the Hugging Face models are cached under it
# the rides and the byte size of each video on the laptop, written by scripts/colab_bundle.py
SIZES = json.loads((DRIVE / 'videos_manifest.json').read_text())
RIDES = list(SIZES)
assert (DRIVE / 'code' / 'bikesafe' / 'infrastructure.py').exists(), f'upload the bikesafe folder as {DRIVE}'
# the videos: in bikesafe/videos, or in the laptop folder backed up by Google Drive for desktop ("Other computers")
SOURCES = [DRIVE / 'videos', *pathlib.Path('/content/drive/Othercomputers').glob('*/bikesafe_videos')]
def find_video(ride):
    return next((d / f'{ride}.mp4' for d in SOURCES if (d / f'{ride}.mp4').exists()), None)
print('video folders:', [str(d) for d in SOURCES])
print('videos missing on Drive:', [r for r in RIDES if find_video(r) is None] or 'none')
shutil.copytree(DRIVE / 'code', ROOT, dirs_exist_ok=True)
# Colab's own torch / torchvision are kept; everything else at the versions the laptop run uses
reqs = [l for l in (ROOT / 'requirements.txt').read_text().splitlines()
        if l.strip() and not l.startswith(('--', 'torch==', 'torchvision=='))]
(ROOT / 'requirements-colab.txt').write_text(''.join(line + chr(10) for line in reqs))
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'opencv-python', 'opencv-contrib-python',
                'opencv-python-headless'])
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-3000:])
assert r.returncode == 0, 'pip install failed (see above)'
print(subprocess.run([sys.executable, '-c', 'import torch, cv2, transformers, ultralytics, open_clip; '
                      'print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "opencv", cv2.__version__)'],
                     capture_output=True, text=True).stdout)

In [ ]:
import json, time
LOCAL_VIDEOS = pathlib.Path('/content/videos')  # read from local disk, not through the Drive mount
LOCAL_VIDEOS.mkdir(exist_ok=True)
env = dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1')
log = DRIVE / 'colab_log.txt'

def finished(ride):
    j = DRIVE / 'work' / 'analysis' / ride / 'infrastructure.json'
    return j.exists() and bool(json.loads(j.read_text()).get('complete'))

def uploaded(ride):
    """The ride's video once Drive has all of it: Drive for desktop lists a file before its content arrives, so the
    size must match the laptop's copy (videos_manifest.json)."""
    src = find_video(ride)
    return src if src is not None and src.stat().st_size == SIZES[ride] else None

pending, waited = [r for r in RIDES if not finished(r)], 0
print('to run:', pending, flush=True)
while pending:  # each ride starts as soon as its video is on Drive, so uploading and computing overlap
    ready = next(((r, s) for r in pending if (s := uploaded(r)) is not None), None)
    if ready is None:
        if waited % 600 == 0:
            print(f'waiting for the videos to finish uploading: {pending}', flush=True)
        time.sleep(60)
        waited += 60
        SOURCES[:] = [DRIVE / 'videos', *pathlib.Path('/content/drive/Othercomputers').glob('*/bikesafe_videos')]
        continue
    r, src = ready
    pending.remove(r)
    dst = LOCAL_VIDEOS / f'{r}.mp4'
    if not (dst.exists() and dst.stat().st_size == SIZES[r]):
        print('copying', src.name, f'{src.stat().st_size / 1e9:.1f} GB', flush=True)
        shutil.copy(src, dst)
    cmd = [sys.executable, '-m', 'bikesafe.infrastructure', str(DRIVE / 'work' / 'perception' / r),
           '--analysis', str(DRIVE / 'work' / 'analysis'), '--videos', str(LOCAL_VIDEOS)]
    with open(log, 'a') as out, subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                                 stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            out.write(line)
            out.flush()
            if 'Loading weights' not in line and line.strip():
                print(line, end='', flush=True)
    print(r, 'exit code', p.returncode, flush=True)
    if p.returncode == 0:
        dst.unlink()  # free the runtime's disk for the next ride
print('all rides done' if all(finished(r) for r in RIDES) else 'some rides did not finish: run all again')

In [ ]:
# type the signs again (from their saved crops and the text already read) wherever they were typed with other
# settings than the current code: a ride passed on the laptop before an update, or after a review round
analysis = sorted(str(j.parent) for j in (DRIVE / 'work' / 'analysis').glob('*/infrastructure.json'))
check = ('import sys, pathlib; from bikesafe.run import signs_outdated; '
         'print(" ".join(pathlib.Path(a).name for a in sys.argv[1:] if signs_outdated(pathlib.Path(a))))')
outdated = subprocess.run([sys.executable, '-c', check, *analysis], cwd=ROOT, capture_output=True, text=True).stdout.split()
print('signs to type again:', outdated or 'none', flush=True)
if outdated:
    cmd = [sys.executable, '-m', 'bikesafe.infrastructure', *[str(DRIVE / 'work' / 'perception' / r) for r in outdated],
           '--analysis', str(DRIVE / 'work' / 'analysis'), '--read-only', '--keep-text']
    p = subprocess.run(cmd, cwd=ROOT, env=env, capture_output=True, text=True)
    for line in (p.stdout + p.stderr).splitlines():
        if 'Loading weights' not in line and 'HF_TOKEN' not in line:
            print(line)
    print('exit code', p.returncode)


In [ ]:
import json
for r in RIDES:
    j = DRIVE / 'work' / 'analysis' / r / 'infrastructure.json'
    info = json.loads(j.read_text()) if j.exists() else {}
    print(r, 'complete:', info.get('complete', False), '| samples', info.get('samples'), '| minutes', info.get('minutes'))